# Values for publication

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import xarray as xr
import dask.dataframe as dd

In [ ]:
# Directories
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir_zen = '../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

In [ ]:
# Show all rows of pandas dataframe
pd.set_option('display.max_rows', None)

---

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='interactive',
                       cores=24,
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='01:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=1)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)
client

---

### Training data: Number of grid cells

In [ ]:
# Get data 
dd_base = dd.read_parquet(f'{dir02}dd_base_r001.parquet')

In [ ]:
# Create dataframe with number of potential training data
dd_base_sel = dd_base[[i for i in dd_base.columns if i.startswith('pot_train')]]
df_sel = dd_base_sel.sum(axis=0).compute()
df_sel_pot_train = {k.replace('pot_train_', ''): v for k, v in df_sel.items()}
df_npot_train = pd.DataFrame([df_sel_pot_train], index=['pot_train'])
df_npot_train

In [ ]:
# Create dataframe with number of training data
dd_base_sel = dd_base[[i for i in dd_base.columns if i.startswith('train_')]]
df_sel = dd_base_sel.sum(axis=0).compute()
df_sel_cveg = {k.replace('train_cveg_', ''): v for k, v in df_sel.items() if 'cveg' in k}
df_sel_csoil = {k.replace('train_csoil_', ''): v for k, v in df_sel.items() if 'csoil' in k}
df_ntrain = pd.DataFrame([df_sel_cveg, df_sel_csoil], index=['train_cveg', 'train_csoil'])
df_ntrain

---

### Training data: Carbon densities

In [ ]:
# Get data 
dd_base = dd.read_parquet(f'{dir02}dd_base_r001.parquet')

In [ ]:
def _get_train_data(ctype, etype):

    """Returns training data"""
    
    # Define cbase (ensure consistency with 02_05_prep_database)
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            cbase = 'cveg_biom'    
        if etype in ['glob_tundr', 'glob_mount']:
            cbase = 'cveg_tund'
        if etype in ['glob_grass', 'glob_past']:
            cbase = 'cveg_gras'
        if etype == 'glob_crop':
            cbase = 'cveg_crop'
    if ctype == 'csoil':
        cbase = 'csoil_000_100'

    # Get training data values
    x = dd_base_sel[dd_base_sel[f'train_{ctype}_{etype}'] == True][cbase].values.compute()
    return pd.Series(x)

In [ ]:
# Prepare dd_base
dd_base_sel = dd_base[
    ['cveg_biom', 'cveg_tund', 'cveg_crop', 'cveg_gras', 'csoil_000_100',
    *[i for i in dd_base.columns if i.startswith('train_')]]] \
    .repartition(partition_size='200 MiB') \
    .persist()

# List of etypes
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount']

# Create dataframe with statistics for training data
rows = []

for ctype in ['cveg', 'csoil']:
    for etype in list_etype:
        s = _get_train_data(ctype, etype)
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        
        rows.append({
            'ctype': ctype,
            'etype': etype,
            'mean': s.mean(),
            'median': s.median(),
            'std': s.std(),
            'q1': q1,
            'q3': q3,
            'min': s.min(),
            'max': s.max(),
            'iqr': iqr,
            'lower_whisker': q1 - 1.5 * iqr,
            'upper_whisker': q3 + 1.5 * iqr
        })

df_training_carbon = pd.DataFrame(rows).round(1)

df_training_carbon

In [ ]:
# Literature values of vegetation carbon
pd.read_excel(f'{dir08}tables/literature_carbon_collection_20260818.xlsx',
              sheet_name='cveg_all')

In [ ]:
pd.read_excel(f'{dir08}tables/literature_carbon_collection_20260818.xlsx',
              sheet_name='csoil_all')

---

### FLUC: Global

In [ ]:
def get_df_fluc_sum():

    """Return dataframe with globally aggregated FLUC values."""

    ds_in = xr.open_dataset(f'{dir_zen}ecosystem_carbon_data_figure_m05.nc')

    # Global sums
    ds_sum = (
        (ds_in[[v for v in ds_in.data_vars if '_fluc_' in v]]
         * ds_in.cell_area)
        .sum(['time', 'lat', 'lon'])
        * 1e-9
    )

    # Preserve variable order from ds_in
    variables = [
        v.removeprefix('chansis_')
        for v in ds_in.data_vars
        if v.startswith('chansis_fluc_')
    ]

    # Convert to dataframe
    s = ds_sum.to_pandas()
    s.index = s.index.str.split('_', n=1, expand=True)
    s.index.names = ['simulation', 'variable']

    df = (
        s.unstack('variable')
        .loc[['chansis', 'cspatial', 'cdgvm'], variables]
        .round(1)
        .reset_index()
        .rename_axis(columns=None)
    )

    return df

df_fluc_sum = get_df_fluc_sum()

df_fluc_sum

In [ ]:
# Difference between Default, cSpatial and cDGVM
def get_fluc_difference_to_default(s0, s1):
    df = df_fluc_sum.set_index('simulation')
    diff = df.loc[s1] - df.loc[s0]

    return pd.DataFrame([
        (diff / df.loc[s0] * 100).round(1),
        diff.round(1),
    ], index=['relative_diff (%)', 'absolute_diff (PgC)'])
    
# Difference between cHansis and cSpatial
get_fluc_difference_to_default('chansis', 'cspatial')

In [ ]:
# Difference between cHansis and cDGVM
get_fluc_difference_to_default('chansis', 'cdgvm')

---

### FLUC: IPCC regions

In [ ]:
# Get and print data
df_fluc_ipcc = pd.read_csv(f'{dir08}tables/table_ipcc_region_fluc_sel.csv')
df_fluc_ipcc

In [ ]:
print(f'Number of IPCC regions: {len(df_fluc_ipcc)}')
print(('Number if IPCC regions with >20 % Difference: ' 
    f'{(df_fluc_ipcc["cspatial_net_diff_per"].abs() > 20).sum().item()}'))

In [ ]:
df_fluc_ipcc.sort_values(
    'cspatial_net_diff_pgc', 
    key=lambda s: s.abs(),
    ascending=False).head(7) \
    [['reg', 'name', 'chansis_net_pgc', 
          'cspatial_net_pgc', 'cspatial_net_diff_pgc', 'cspatial_net_diff_per']]

In [ ]:
df_fluc_ipcc.sort_values('cspatial_net_diff_per', ascending=True).head(5) \
    [['reg', 'name', 'chansis_net_pgc', 
      'cspatial_net_pgc', 'cspatial_net_diff_pgc', 'cspatial_net_diff_per']] 

In [ ]:
df_fluc_ipcc.sort_values('cspatial_net_diff_per', ascending=False).head(5) \
    [['reg', 'name', 'chansis_net_pgc', 
      'cspatial_net_pgc', 'cspatial_net_diff_pgc', 'cspatial_net_diff_per']] 

---

### Carbon pools: BLUE, TRENDY, Literature

In [ ]:
# Get data
df_cstocks = pd.read_csv(f'{dir_zen}ecosystem_carbon_data_figure_m06.csv')
df_cstocks

In [ ]:
# Calculate difference to reference value
for source in ['chansis_2022', 'cspatial_2022', 'cdgvm_2022']:
    
    print(f'Difference {source} vs. reference:')

    for carbon_type in ['cveg', 'csoil']:

        v_s = df_cstocks[
            (df_cstocks.source == f'blue_{source}') & 
            (df_cstocks.carbon_type == carbon_type)]['value_pgc'].item()
        v_ref = df_cstocks[
            (df_cstocks.source == 'reference') & 
            (df_cstocks.carbon_type == carbon_type)]['value_pgc'].item()
        
        diff_pgc = v_s - v_ref
        diff_per = (v_s / v_ref - 1) * 100 
        
        print(f'- {carbon_type}: {round(diff_pgc)} PgC ({round(diff_per)}%)')

---

In [ ]:
cluster.close()